<a href="https://colab.research.google.com/github/masaki-kawa/uts-study-notes/blob/main/data/raw/colab/Deep_Learning_Lab6_Exercise2_Solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNN with Pytorch

# Multi-Class Classification of News Articles

In this exercise, we will use a sequence model (GRU) to analyse text from news articles and predict their topic. We will be using the famous AG news dataset. It is a collection of more than 1 million news articles published by Xiang Zhang.

## Objective

Our goal is to train a bi-directional GRU model with embeddings and assess its performance.

## Instructions

This is a guided exercise where some of the code have already been pre-defined. Your task is to fill the remaining part of the code (it will be highlighted with placehoders) to train and evaluate your model.

This exercise is split in several parts:
1.   Loading and Exploration of the Dataset
2.   Creating the vocabulary
3.   Preparing the Dataset
4.   Defining the Architecture of the GRU with embeddings
5.   Training and Evaluation of the Model
6.   Analysing the Results

### 1. Loading and Exploration of the Dataset

**[1.1]** First we need to import the necessary libraries

In [ ]:
!pip uninstall -y torch torchtext torchvision
!pip install torch==2.3.0 torchtext==0.18.0 torchvision==0.18.0

In [ ]:
# Solution
import torch
import torch.nn as nn
from torchtext.data.utils import get_tokenizer
import spacy
from torchtext.vocab import build_vocab_from_iterator
from itertools import chain
from torchtext.data.functional import to_map_style_dataset
from torch.nn.utils.rnn import pad_sequence
from torchtext.vocab import Vocab
from torch.utils.data import DataLoader
import torch.optim as optim
from sklearn.model_selection import train_test_split
from google.colab import files
import matplotlib.pyplot as plt
import seaborn as sns
import time

**[1.2]** Enable Synchronous CUDA Operations

Description: Set CUDA_LAUNCH_BLOCKING to '1' to make all CUDA operations synchronous.

Purpose: This setting helps in debugging by forcing CUDA to perform operations synchronously, which provides a more precise error traceback.

In [ ]:
# Solution
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

**[1.3]** Now we will choose the device type [CPU/GPU] in such a way that if GPU is available then use GPU otherwise use CPU.

In [ ]:
# Solution
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

**[1.4]** Now we will get the data directly from kaggle. To do that we need to upload `kaggle.json` file, downloaded from kaggle while being signed in by using **own account**.

How to get The Kaggle.json:

1. Sign in to your kaggle account and go to the following link `https://www.kaggle.com/settings/account`
2. Scroll down to API
3. Create new token
4. It will download a kaggle.json for you
5. upload the kaggle.json after you run the following cell.

Finally upload the `kaggle.json`

In [ ]:
# Solution
files.upload()

**[1.5]** Let's connect Kaggle to Google Colab

In [ ]:
# Solution
!mkdir -p ~/.kaggle # Creates a .kaggle directory, if it does not exist. -p flag is there to make sure there is no error if the directory already exists
!cp kaggle.json ~/.kaggle/ # Copies the kaggle.json, which contains the API credentials to the .kaggle directory. this file is used to get API access.
!chmod 600 ~/.kaggle/kaggle.json  # This changes the permission to read/write for the file. This ensures security measures to protect API credentials.

**[1.5]** Now download the dataset from kaggle: https://www.kaggle.com/datasets/amananandrai/ag-news-classification-dataset

In [ ]:
# Solution
!kaggle datasets download -d amananandrai/ag-news-classification-dataset

**[1.6]** Unzip the downloaded dataset

In [ ]:
# Solution
!unzip /content/ag-news-classification-dataset.zip

**[1.7]** Now we will read the data using a `read_csv` function of pandas and save it into panda's dataframe named `train_data` and `test_data`.

In [ ]:
# Solution
import pandas as pd
train_data = pd.read_csv('/content/train.csv')
test_data = pd.read_csv('/content/test.csv')
print(train_data.head(2))

**[1.8]** Let's have a look at the shape of the data

In [ ]:
# Solution
print(f"Train data - Number of Rows: {train_data.shape[0]}")
print(f"Train data - Number of Columns: {train_data.shape[1]}")
print("\n")
print(f"Test data - Number of Rows: {test_data.shape[0]}")
print(f"Test data - Number of Columns: {test_data.shape[1]}")

**[1.9]** Check the name of columns of train_data

In [ ]:
# Solution
train_data.columns

**[1.10]** Let's change the column from `Class Index` to `class_index`, `Title` to `title`, and `Description` to `description`.

In [ ]:
# Solution
train_data = train_data.rename(columns={
    'Class Index': 'class_index',
    'Title': 'title',
    'Description': 'description'
})

test_data = test_data.rename(columns={
    'Class Index': 'class_index',
    'Title': 'title',
    'Description': 'description'
})

**[1.11]** Now Check the unique class values of `train_data`.

In [ ]:
# Solution
train_data['class_index'].unique()

**[1.12]** Now subtract 1 from each value of unique values of `train_data` and `test_data`.

purpose: For multiclass classification, pytorch expects the class values to start from 0. From the dataset, it is apparent that the class values start at 1 and ends at 4. Subtracting 1 from each value will maintain the integrity of the data while serving our purpose.

In [ ]:
# Solution
train_data['class_index'] = train_data['class_index'] - 1
test_data['class_index'] = test_data['class_index'] - 1

**[1.13]** After subtracting, let's print the unique values in `train_data` and `test_data`.

In [ ]:
# Solution
print(f"Unique Values in Train Data: {train_data['class_index'].unique()}")
print(f"Unique Values in Test Data: {test_data['class_index'].unique()}")

**[1.14]** To visualize the distribution of  train and test data let's use the countplot function of seaborn (https://seaborn.pydata.org/generated/seaborn.countplot.html)

In [ ]:
# Solution
# Class distribution in train data
sns.countplot(x=train_data['class_index'])
plt.title('Distribution of Classes for train data')
plt.show()

In [ ]:
# Solution
# Class distribution in test data
sns.countplot(x=test_data['class_index'])
plt.title('Distribution of Classes for test data')
plt.show()

### 2.   Preparing the Dataset

**[2.1]** Assign the `train_data.description` to X_train, `test_data.description` to X_test, `train_data.class_index` to y_train, `test_data.class_index` to y_test.

In [ ]:
# Solution
X_train = train_data.description
X_test = test_data.description
y_train = train_data.class_index
y_test = test_data.class_index

**[2.2]** **Tokenization**

Loading the English tokenizer from spaCy of Python (https://spacy.io/usage/models) for tokenizing the text. Note: spaCy is a free, open-source library for advanced Natural Language Processing (NLP) in Python.


In [ ]:
# Solution
# Load spaCy English tokenizer
spacy_en = spacy.load('en_core_web_sm')

**[2.3]** Let's use the lambda function of Python that takes a piece of text and tokenizes it into words. Save the outcome into a variable called `tokenizer`.

In [ ]:
# Solution
# The lambda function takes a piece of text and tokenizes it into words.
tokenizer = lambda text: [tok.text for tok in spacy_en.tokenizer(text)]

**[2.4]** **Build Vocabulary**

This step creates vocabulary from the tokenized text data, which is necessary for converting words to integers. For defining the vocabulary  size explicitely, set the `max_vocab_size` = 10000.

In [ ]:
# Solution
# Define vocabulary size explicitly
max_vocab_size = 10000 # prevents the vocabulary from becoming too large and helps manage memory.

vocab = build_vocab_from_iterator((tokenizer(text) for text in X_train), specials=["<unk>", "<pad>"], max_tokens=max_vocab_size)
vocab.set_default_index(vocab["<unk>"])  # Handling unknown tokens

**[2.5]** Now our vocabulary is ready. Let's print the size of the vocabulary.

In [ ]:
# Solution
# Now taking a look at the vocabs to check if everything is in order.
vocab_size = len(vocab)
print("Vocabulary size:", vocab_size)

**[2.6]** Custom data collation function named `collate_batch(batch)`

**Task** : prepares batches of text data by tokenizing, converting to numerical indices, and padding them to a uniform length of sequences (https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pad_sequence.html), which is necessary for batch processing in neural networks. It also encodes labels to numerical values suitable for model training. The tasks are as follows:
1. Read text and label in batch
2. Tokenize text using a predefined tokenizer
3. Convert to a list of vocabulary indices
4. Convert list of indices to a PyTorch tensor
5. Pad the text sequences using 'pad_sequence' of PyTorch (https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pad_sequence.html)
6. Convert labels to a tensor

In [ ]:
# Solution
# Initialize a flag to control the printing
print_flag = True

def collate_batch(batch):
    global print_flag
    # Initialize lists to hold processed texts and labels
    processed_texts = []
    labels = []

    # Process each text and corresponding label in the batch
    for text, label in batch:
        # Tokenize text using a predefined tokenizer and convert to a list of vocabulary indices
        tokenized_text = tokenizer(text)
        numerical_text = [vocab[token] for token in tokenized_text if token in vocab]

        # Convert list of indices to a PyTorch tensor and append to the list of processed texts
        tensor_text = torch.tensor(numerical_text, dtype=torch.int64)
        processed_texts.append(tensor_text)

        # Directly append the label as it is already an integer
        labels.append(label)

        # Check if should print
        if print_flag:
            print("Sample text:", text)
            print("Tokenized text:", tokenized_text)
            print("Numerical text:", numerical_text)
            print_flag = False  # Set the flag to False after printing once

    # Pad the text sequences and convert labels to a tensor
    padded_texts = pad_sequence(processed_texts, padding_value=vocab['<pad>'], batch_first=True)
    labels_tensor = torch.tensor(labels, dtype=torch.int64)

    return padded_texts, labels_tensor


**[2.7]** Now we will call the DataLoader function that iteratively loads data based on batch size, shuffle and save it into two different variables called `train_loader`, and `test_loader`. Set the `BATCH_SIZE` to 32. Inside the DataLoader function we will call `collate_batch` function.

In [ ]:
# Solution
# Creating DataLoader for each dataset split
train_loader = DataLoader(list(zip(X_train, y_train)), batch_size=32, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(list(zip(X_test, y_test)), batch_size=32, shuffle=False, collate_fn=collate_batch)

### 3.   Defining the Architecture of the GRU with embeddings

**[3.1]** Let's create a Bi-directional GRU model named `TextClassifier`. This GRU model processes sequences of data, applying an 1. embedding layer, 2. GRU layer, 3. dropout and 4. fully connected layer for output predictions.

In [ ]:
# Solution
class TextClassifier(nn.Module):
    """
    A text classification model using GRU (Gated Recurrent Unit) for processing sequences.

    This model incorporates a word embedding layer, a bidirectional GRU for capturing dependencies
    in both directions of the sequence, and a fully connected layer for classification.

    Attributes:
        embedding (nn.Embedding): Embeds input vocabulary into dense vectors.
        gru (nn.GRU): A bidirectional GRU layer that processes the embedded word vectors.
        dropout (nn.Dropout): Dropout layer to reduce overfitting by randomly setting
                              a fraction of input units to 0 at each update during training.
        fc1 (nn.Linear): First fully connected layer that maps GRU outputs to an intermediate dimension.
        fc2 (nn.Linear): Second fully connected layer that maps intermediate representations to class scores.

    Parameters:
        vocab_size (int): Size of the input vocabulary.
        embedding_size (int): Dimension of the embedding vectors.
        hidden_dim (int): Number of features in the hidden state of the GRU.
        n_classes (int): Number of target classes for classification.
        dropout_rate (float): Dropout rate used in the dropout layer.
    """
    def __init__(self, vocab_size, embedding_size, hidden_dim, n_classes, dropout_rate=0.5):
        super(TextClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size, padding_idx=0)
        self.gru = nn.GRU(embedding_size, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(2 * hidden_dim, 16)  # 2 is used because the GRU layer is bidirectional.
                                                  # Bidirectional GRU processes the input sequence in opposite directions
                                                  # - one forward, one backward
                                                  # Each GRU produces a hidden state of the size of hidden_dim
        self.fc2 = nn.Linear(16, n_classes)

    def forward(self, x):
        """
        Forward pass of the classifier.

        Args:
            x (torch.Tensor): Input tensor containing batches of tokenized text.

        Returns:
            torch.Tensor: A tensor containing the softmax probabilities for class predictions.
        """
        x = self.embedding(x)  # Convert input words to their embedding representations, because of the embedding layer, we did not require to use vectorization explicitly.
        x, _ = self.gru(x)     # Process sequence via GRU
        x = self.dropout(x)    # Apply dropout to reduce overfitting
        x = torch.relu(self.fc1(x[:, -1, :]))  # Apply ReLU activation to the output of the last time step
        x = self.fc2(x)        # Final classification layer
        return torch.softmax(x, dim=1)  # Use softmax to convert logits to probabilities

**[3.2]** Let's set the model parameters. Set the embedding dimension as 128, hidden dimension as 256, dropout as 0.5, set the number of epochs as 5 and number of classes as 4 (since this is multi classification).

In [ ]:
# Solution
# Parameters for the model
vocab_size = 10000
embedding_size = 128
hidden_dim = 256
n_classes = 4
num_epochs = 5

**[3.3]** Let's instantiate the newly created LSTMModel and save it to `model`. Finally send the model to available device (CPU/GPU)

In [ ]:
# Solution
model = TextClassifier(vocab_size, embedding_size, hidden_dim, n_classes)
model.to(device)  # Move to GPU if available

**[3.4]** Print the model summary

In [ ]:
# Solution
print(model)

### 4.   Training and Evaluation of the Model

**[4.1]**  Instantiate a `nn.CrossEntropyLoss()` and save it into a variable called `criterion`. After then Instantiate a `optim.Adam()` optimizer with the model's parameters and 0.001 as learning rate and save it into a variable called `optimizer`.

In [ ]:
# Solution
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

**[4.2]** Define the empty lists named `train_losses`, `train_accuracies`, `val_losses`, and `val_accuracies` to hold the values of accuracies and losses while training

In [ ]:
# Solution
# Lists to hold metric values for visualization and analysis
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

**[4.3]** Let's define a function called `train_model` that takes the following parameters model, device, train_loader, optimizer, criterion, device, epoch and writer for TensorBoard. This function is responsible to train our newly created GRU model.
A nested loop is initiated that extracts data from `train_loader` and introduce the following logics:
- reset the gradients (https://pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)
- perform the forward propagation and get the model predictions
- calculate the loss between the predictions and the actuals
- perform back propagation
- update the weights
- Count the total loss

In [ ]:
# Solution
def train_model(model, train_loader, optimizer, criterion, device, epoch, writer):
    """
    Train the model for one epoch over the training dataset.

    Args:
        model (torch.nn.Module): The neural network model to be trained.
        train_loader (torch.utils.data.DataLoader): DataLoader for the training data.
        optimizer (torch.optim.Optimizer): Optimizer used to update model weights.
        criterion (torch.nn.Module): Loss function used for training.
        device (torch.device): Device to run the training on (e.g., 'cpu' or 'cuda').
        epoch (int): Current epoch number, used for logging.
        writer (torch.utils.tensorboard.SummaryWriter): Writer object for logging to TensorBoard.

    Returns:
        float: Average loss for this epoch.
        float: Accuracy for this epoch.
    """

    # Set the model to training mode (this enables features like dropouts)
    model.train()

    # Initialize metrics
    total_loss = 0
    correct_predictions = 0
    total_samples = 0

    # Iterate over batches of data from the DataLoader
    for inputs, labels in train_loader:
        # Move data to the specified device (GPU or CPU)
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Reset gradients (this is necessary in each step)
        optimizer.zero_grad()

        # Forward pass: compute the model output
        outputs = model(inputs)

        # Compute loss: difference between the predicted outputs and actual labels
        loss = criterion(outputs, labels)

        # Backward pass: compute gradient of the loss with respect to model parameters
        loss.backward()

        # Perform a single optimization step (parameter update)
        optimizer.step()

        # Accumulate the training loss
        total_loss += loss.item()
        _, predicted = torch.max(outputs, dim=1)

        # Count correct predictions
        correct_predictions += (predicted == labels).sum().item()

        # Update the total number of samples seen
        total_samples += labels.size(0)

    # Calculate average loss and accuracy over all training samples
    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / total_samples

    # Log training loss and accuracy for each epoch to TensorBoard
    writer.add_scalar('Loss/Train', avg_loss, epoch)
    writer.add_scalar('Accuracy/Train', accuracy, epoch)

    return avg_loss, accuracy

**[4.4]** To validate/evaluate the model on validation/test data, a nested loop is initiated that extracts data from `data_loader` and introduce the following logics:
- disable computing gradients (https://pytorch.org/docs/stable/generated/torch.no_grad.html)
- perform the forward propagation and get the model predictions
- calculate the loss between the predictions and the actuals
- Count the total loss
- Count the correct outcome

In [ ]:
# Solution
def evaluate_model(model, data_loader, criterion, device, writer, epoch):
    """
    Evaluate the model on a given dataset.

    Args:
        model (torch.nn.Module): The neural network model to be evaluated.
        data_loader (torch.utils.data.DataLoader): DataLoader for the dataset to be evaluated.
        criterion (torch.nn.Module): Loss function used for evaluation.
        device (torch.device): Device to run the evaluation on (e.g., 'cpu' or 'cuda').
        writer (torch.utils.tensorboard.SummaryWriter): Writer object for logging to TensorBoard.
        epoch (int): Current epoch number, used for logging.

    Returns:
        float: Average loss over the evaluated dataset.
        float: Accuracy over the evaluated dataset.
    """

    # Set model to evaluation mode. This will turn off features like Dropout.
    model.eval()

    # Disable gradient computation; gradients are not needed for evaluation, which saves memory and computation.
    with torch.no_grad():
        total_loss = 0
        correct_predictions = 0
        total_samples = 0
        predictions = []
        actuals = []

        # Iterate over all batches in the data loader
        for inputs, labels in data_loader:
            # Move data to the specified device
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Forward pass: compute the model output
            outputs = model(inputs)

            # Compute loss: evaluate the difference between output and true labels
            loss = criterion(outputs.squeeze(), labels)

            # Accumulate the loss over each batch
            total_loss += loss.item()
            _, predicted = torch.max(outputs, dim=1)

            # Reshape predictions to a flat array, transfer to CPU, and convert to numpy for further processing.
            predictions.extend(predicted.view(-1).cpu().numpy())
            # Similarly, process actual labels.
            actuals.extend(labels.view(-1).cpu().numpy())

            # Count correct predictions
            correct_predictions += (predicted == labels).sum().item()

            # Update total number of samples processed
            total_samples += labels.size(0)

        # Calculate average loss and accuracy over all samples
        avg_loss = total_loss / len(data_loader)
        accuracy = correct_predictions / total_samples

        # Log validation loss and accuracy for this epoch to TensorBoard
        writer.add_scalar('Validation Loss', avg_loss, epoch)
        writer.add_scalar('Validation Accuracy', accuracy, epoch)

    return avg_loss, accuracy, predictions, actuals

**[4.4]** Set up TensorBoard for logging and visualizing the neural network training process.

In [ ]:
# Solution
from torch.utils.tensorboard import SummaryWriter

# Purpose: Set up TensorBoard for logging and visualizing the neural network training process.
# This allows for effective monitoring and analysis of the model performance over time, aiding in debugging and optimization.

# Initialize the TensorBoard writer:
# The 'SummaryWriter' object is responsible for writing data to a directory ('runs/imdb_classification') that TensorBoard will read from.
writer = SummaryWriter('runs/imdb_classification')

# Explanation:
# - 'SummaryWriter' is a class from the 'torch.utils.tensorboard' package.
# - The directory specified ('runs/imdb_classification') is where all logs will be saved. This directory structure is useful for organizing multiple experiments.
# - TensorBoard will access this directory to display metrics such as loss and accuracy, along with other visualizable data, helping to track the model's training and validation progress over epochs.

**[4.6]** **Training:** Now it is time to train our model by calling `train_model` and validate it through `evaluate_model` functions that introduced previously. The evaluation metrics are stored in train_losses, val_losses, train_accuracies, val_accuracies by using append function.

In [ ]:
# Solution
import time  # Importing the time module to track the duration of each epoch

# Loop through each epoch to train and validate the model
for epoch in range(num_epochs):
    # Record the start time of the epoch to calculate the duration later
    start_time = time.time()

    # Train the model using the train_loader
    # train_model function returns the average loss and accuracy for this training epoch
    # train_model(model, train_loader, optimizer, criterion, device, epoch, writer):
    train_loss, train_accuracy = train_model(model, train_loader, optimizer, criterion, device, epoch, writer)

    # Evaluate the model using the val_loader
    # evaluate_model function returns the average loss and accuracy for this validation epoch
    val_loss, val_accuracy, val_predictions, val_actuals = evaluate_model(model, test_loader, criterion, device, writer, epoch)

    # Store metrics for later analysis and visualization in TensorBoard
    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    # Record the end time of the epoch and calculate the duration
    end_time = time.time()
    epoch_duration = end_time - start_time

    # Print a detailed summary of the epoch's results to the console
    print('*' * 50)
    print(f'Epoch: {epoch + 1}')
    print(f'Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}')
    print(f'Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}')
    print(f'Epoch Duration: {epoch_duration:.2f} seconds')  # Duration of the epoch in seconds
    print('*' * 50)

### 5. Analysing the results

**[5.1]** Load tensorboard to visualize the training and validation

In [ ]:
# Solution
%load_ext tensorboard
%tensorboard --logdir runs

**[5.2]** Let's plot the training and validation losses

In [ ]:
# Solution
# Plotting the training and validation loss
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

**[5.3]** Let's plot the training and validation accuracies

In [ ]:
# Solution
# Plotting the training and validation loss
plt.figure(figsize=(10, 5))
plt.plot(train_accuracies, label='Training Accuracy')
plt.plot(val_accuracies, label='Validation Accuracy')
plt.title('Training and Validation Accuracies Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
# Solution
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(val_actuals, val_predictions)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()